<h1><b><i>SCRIPT PRINCIPAL</b></i></h1>
<h2><b><i>Importação de Bibliotécas</b></i></h2>

In [43]:
import sys
sys.path.append(r'C:\pylibs')

import pandas as pd
import numpy as np
import openpyxl

In [44]:
import pandas as pd
import openpyxl
import numpy as np
import os

In [45]:
# ==============================================================================
# 1. SETUP INICIAL E CARREGAMENTO
#    (Melhor prática: definir constantes e carregar o arquivo de forma segura)
# ==============================================================================

# Definição do caminho do arquivo em uma variável para fácil manutenção
FILE_PATH = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx'
OUTPUT_DIR = 'Dados Gerados'

# Verifica se o arquivo existe antes de tentar carregá-lo
if not os.path.exists(FILE_PATH):
    print(f"ERRO: O arquivo '{FILE_PATH}' não foi encontrado.")
    # Encerra o script ou lança uma exceção se o arquivo principal não existir
    exit()

# Cria a pasta de saída se ela não existir
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Arquivo '{FILE_PATH}' encontrado. Iniciando processamento...")

Arquivo 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx' encontrado. Iniciando processamento...


In [46]:
# %% [markdown]
# <h1><b><i>SCRIPT PRINCIPAL UNIFICADO – VERSÃO FINAL</i></b></h1>
# <h2><b><i>Processamento automático de todas as abas do PPI</i></b></h2>

# %%
import sys
sys.path.append(r'C:\pylibs')  # (opcional)

import pandas as pd
import numpy as np
import openpyxl
import os
from collections import defaultdict

# ==============================================================================
# 1. FUNÇÕES AUXILIARES
# ==============================================================================

def expandir_mescladas_para_cabecalho(df_cab):
    """Expande horizontalmente (ffill) para simular células mescladas."""
    return df_cab.ffill(axis=1)

def montar_cabecalho_hierarquico(df_cab):
    """
    Recebe um DataFrame com N linhas de cabeçalho (já com ffill)
    e retorna uma lista de nomes de colunas concatenando os níveis.
    """
    lixo = {"#REF!", "TOTAL", "SOMA", 0, "0", None, ""}
    df_cab = df_cab.replace(list(lixo), pd.NA)
    
    colunas = []
    for col in range(df_cab.shape[1]):
        partes = []
        for row in range(df_cab.shape[0]):
            valor = df_cab.iat[row, col]
            if pd.isna(valor):
                continue
            valor = str(valor).strip()
            if not valor or valor.startswith(("=", "#")):
                continue
            if valor.upper() in ("TOTAL", "SOMA"):
                continue
            try:
                float(valor.replace(",", "."))
                continue
            except:
                pass
            partes.append(valor)
        colunas.append(" - ".join(partes) if partes else "")
    
    # Remove duplicatas adicionando sufixo __n
    contador = defaultdict(int)
    novas = []
    for nome in colunas:
        contador[nome] += 1
        if contador[nome] == 1:
            novas.append(nome)
        else:
            novas.append(f"{nome}__{contador[nome]}")
    return novas

def processar_aba(file_path, sheet_name, n_linhas_cabecalho, keywords_id,
                  separador=" - ", remover_totais=True, valor_minimo=0):
    """
    Processa uma aba com cabeçalho hierárquico e retorna o DataFrame no formato:
    ID + identificadores + Tipo + Período + Métricas (pivotadas).
    """
    print(f"  Processando '{sheet_name}'...", end=" ")
    
    # 1) Verificar se a aba existe
    try:
        xl = pd.ExcelFile(file_path)
        if sheet_name not in xl.sheet_names:
            print(f"❌ Aba não encontrada!")
            return None
    except:
        print(f"❌ Erro ao ler o arquivo!")
        return None

    # 2) Ler cabeçalho e dados
    cab = pd.read_excel(file_path, sheet_name=sheet_name, header=None, nrows=n_linhas_cabecalho)
    dados = pd.read_excel(file_path, sheet_name=sheet_name, header=None, skiprows=n_linhas_cabecalho)

    # 3) Detectar última coluna com dados no cabeçalho
    ultima_cab = cab.columns[cab.notna().any(axis=0)].max()
    if pd.isna(ultima_cab):
        ultima_cab = -1

    # 4) Detectar última coluna com dados nas primeiras 10 linhas dos dados
    amostra = dados.iloc[:10, :]
    ultima_dados = amostra.columns[amostra.notna().any(axis=0)].max()
    if pd.isna(ultima_dados):
        ultima_dados = -1

    # 5) Usar o máximo entre as duas + 1
    ultima_col = max(ultima_cab, ultima_dados) + 1
    ultima_col = int(ultima_col)
    if ultima_col <= 0:
        ultima_col = cab.shape[1]

    # 6) Cortar
    cab = cab.iloc[:, :ultima_col]
    dados = dados.iloc[:, :ultima_col]

    print(f"(colunas: {ultima_col})", end=" ")

    # 7) Gerar nomes
    cab_exp = expandir_mescladas_para_cabecalho(cab)
    nomes_colunas = montar_cabecalho_hierarquico(cab_exp)

    # 8) Ajustar tamanho
    if len(nomes_colunas) != dados.shape[1]:
        if len(nomes_colunas) > dados.shape[1]:
            nomes_colunas = nomes_colunas[:dados.shape[1]]
        else:
            nomes_colunas += [f"Unnamed_{i}" for i in range(dados.shape[1] - len(nomes_colunas))]

    dados.columns = nomes_colunas
    dados = dados.dropna(how='all')

    # 9) Identificar colunas de ID
    id_cols = []
    for col in dados.columns:
        col_upper = col.upper()
        if any(kw.upper() in col_upper for kw in keywords_id):
            id_cols.append(col)

    if id_cols:
        dados[id_cols] = dados[id_cols].ffill()

    # 10) Remover totais
    if remover_totais:
        metric_cols = [c for c in dados.columns if c not in id_cols]
        if metric_cols:
            mask_total = dados[metric_cols].astype(str).apply(
                lambda row: row.str.contains('TOTAL|SOMA', case=False, na=False).any(), axis=1
            )
            dados = dados[~mask_total].reset_index(drop=True)

    metric_cols = [c for c in dados.columns if c not in id_cols]
    if not metric_cols:
        print("⚠️ Sem colunas de métrica. Retornando apenas identificadores.")
        return dados

    # 11) Melt
    df_long = dados.melt(id_vars=id_cols, value_vars=metric_cols,
                         var_name="cabecalho_completo", value_name="Valor")
    df_long = df_long.dropna(subset=["Valor"])
    if valor_minimo is not None:
        df_long = df_long[df_long["Valor"] != valor_minimo]

    # 12) Separar níveis – VERSÃO SIMPLIFICADA E ROBUSTA
    split_headers = df_long["cabecalho_completo"].str.split(separador, expand=True)
    num_niveis = split_headers.shape[1]

    if num_niveis == 0:
        df_long["Metrica"] = df_long["cabecalho_completo"]
        df_long["Tipo"] = "Geral"
        df_long["Período"] = "Geral"
    elif num_niveis == 1:
        df_long["Metrica"] = split_headers[0]
        df_long["Tipo"] = "Geral"
        df_long["Período"] = "Geral"
    elif num_niveis == 2:
        df_long["Tipo"] = split_headers[0]
        df_long["Período"] = "Geral"
        df_long["Metrica"] = split_headers[1]
    else:
        # 3 ou mais níveis
        # Primeiros (num_niveis - 2) = Tipo (concatenados)
        # Penúltimo = Período
        # Último = Métrica
        colunas_tipo = [split_headers[i] for i in range(num_niveis - 2)]
        # Concatena com o separador, ignorando valores NaN e convertendo para string
        # Usa apply com fillna('') para evitar erros
        df_long["Tipo"] = pd.concat(colunas_tipo, axis=1).apply(
            lambda row: separador.join(row.fillna('').astype(str).tolist()), axis=1
        )
        # Remove espaços extras e separadores duplicados
        df_long["Tipo"] = df_long["Tipo"].str.replace(r'\s+', ' ', regex=True).str.strip()
        df_long["Tipo"] = df_long["Tipo"].str.replace(r' - - ', ' - ', regex=False)
        df_long["Tipo"] = df_long["Tipo"].str.replace(r' - $', '', regex=True)
        df_long["Período"] = split_headers[num_niveis - 2]
        df_long["Metrica"] = split_headers[num_niveis - 1]

    df_long.drop(columns=["cabecalho_completo"], inplace=True)

    # 13) Pivot
    index_cols = id_cols + ["Tipo", "Período"]
    df_pivot = df_long.pivot_table(
        index=index_cols,
        columns="Metrica",
        values="Valor",
        aggfunc="first"
    ).reset_index()
    df_pivot.columns = [col if col == '' else col for col in df_pivot.columns]

    print("✅")
    return df_pivot

# ==============================================================================
# 2. SETUP INICIAL
# ==============================================================================

# ALTERE O CAMINHO PARA O SEU ARQUIVO, SE NECESSÁRIO
FILE_PATH = 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx'
OUTPUT_DIR = 'Dados Gerados'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📂 Arquivo: '{FILE_PATH}'")
if not os.path.exists(FILE_PATH):
    print("❌ Arquivo não encontrado!")
    exit()

print("🚀 Iniciando processamento...\n")

# ==============================================================================
# 3. CONFIGURAÇÕES DAS ABAS
# ==============================================================================

configs = {
    'INFORMAÇÕES (1)': {
        'n_linhas': 4,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR', 'ANO LEILÃO', 'DATA DE INÍCIO',
                     'ANO DA CONCESSÃO', 'PRAZO', 'CAPEX', 'OPEX', 'INVESTIMENTO TOTAL',
                     'TRECHO', 'KM INICIAL', 'KM FINAL', 'EXTENSÃO', 'SITUAÇÃO']
    },
    'PER (1)': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR']
    },
    'PLAN_EXEC ATÉ ANO ANTERIOR (22)': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR']
    },
    'META(1) 2023': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR']
    },
    'META(2024)': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR']
    },
    'META(2025)': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR']
    },
    'META(2026)': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR']
    },
    'META(2027)': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR']
    },
    'iNEXECUTADO (pendências até 24)': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR']
    },
    'À EXECUTAR (2026 DIANTE)': {
        'n_linhas': 3,
        'keywords': ['SETOR', 'UF', 'ESTADO/LOTE', 'BR', 'EMPREENDIMENTO', 'PROPONENTE',
                     'EXECUTOR', 'ESTRUTURADOR', 'OBSERVAÇÕES', 'SITUAÇÃO GERAL']
    }
}

# ==============================================================================
# 4. PROCESSAR CADA ABA
# ==============================================================================

processed_dfs = {}

for sheet_name, params in configs.items():
    df_proc = processar_aba(
        file_path=FILE_PATH,
        sheet_name=sheet_name,
        n_linhas_cabecalho=params['n_linhas'],
        keywords_id=params['keywords']
    )
    if df_proc is not None:
        processed_dfs[sheet_name] = df_proc
        # Salvar individualmente
        nome_arquivo = sheet_name.replace('/', '_').replace(' ', '_') + '.xlsx'
        caminho = os.path.join(OUTPUT_DIR, nome_arquivo)
        df_proc.to_excel(caminho, index=False)
        print(f"    💾 Salvo em: {caminho}")
    else:
        print(f"    ❌ Falha ao processar '{sheet_name}'.")

print("\n" + "="*60)

# ==============================================================================
# 5. CONCATENAÇÃO ESPECIAL: PLAN_EXEC + META(2023) + META(2024)
# ==============================================================================

print("\n🔗 Concatenando PLAN_EXEC + META(2023) + META(2024)...")

keys = ['PLAN_EXEC ATÉ ANO ANTERIOR (22)', 'META(1) 2023', 'META(2024)']
dfs = []
for key in keys:
    if key in processed_dfs:
        dfs.append(processed_dfs[key])
    else:
        print(f"⚠️ '{key}' não processado, ignorado.")

if dfs:
    all_cols = set()
    for df in dfs:
        all_cols.update(df.columns)
    all_cols = sorted(all_cols)
    
    dfs_alinhados = [df.reindex(columns=all_cols) for df in dfs]
    df_concat = pd.concat(dfs_alinhados, axis=0, ignore_index=True)
    
    caminho_concat = os.path.join(OUTPUT_DIR, 'PLAN_EXEC_ATÉ_ANO_ANTERIOR_(24).xlsx')
    df_concat.to_excel(caminho_concat, index=False)
    print(f"✅ Concatenado e salvo em: {caminho_concat}")
else:
    print("❌ Nenhum DataFrame disponível para concatenação.")

# ==============================================================================
# 6. GERAR ARQUIVOS EXTRAS (2025(2).xlsx e PER(2).xlsx)
# ==============================================================================

print("\n📄 Gerando arquivos extras...")

if 'META(2025)' in processed_dfs:
    df_2025 = processed_dfs['META(2025)']
    df_2025.to_excel('2025(2).xlsx', index=False)
    print("✅ 2025(2).xlsx gerado.")
else:
    print("⚠️ META(2025) não disponível para gerar 2025(2).xlsx.")

if 'PER (1)' in processed_dfs:
    df_per = processed_dfs['PER (1)']
    df_per.to_excel('PER(2).xlsx', index=False)
    print("✅ PER(2).xlsx gerado.")
else:
    print("⚠️ PER (1) não disponível para gerar PER(2).xlsx.")

# ==============================================================================
# 7. FINALIZAÇÃO
# ==============================================================================

print("\n🎯 Processamento concluído com sucesso!")
print(f"Todos os arquivos estão em '{OUTPUT_DIR}/'.")

📂 Arquivo: 'Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21.xlsx'
🚀 Iniciando processamento...

  Processando 'INFORMAÇÕES (1)'... (colunas: 20) ✅
    💾 Salvo em: Dados Gerados\INFORMAÇÕES_(1).xlsx
  Processando 'PER (1)'... (colunas: 272) ✅
    💾 Salvo em: Dados Gerados\PER_(1).xlsx
  Processando 'PLAN_EXEC ATÉ ANO ANTERIOR (22)'... 

c:\Users\carlos.ramos\OneDrive - Presidência da República\Área de Trabalho\Planilhas\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


(colunas: 69) ✅
    💾 Salvo em: Dados Gerados\PLAN_EXEC_ATÉ_ANO_ANTERIOR_(22).xlsx
  Processando 'META(1) 2023'... 

c:\Users\carlos.ramos\OneDrive - Presidência da República\Área de Trabalho\Planilhas\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


(colunas: 238) ✅
    💾 Salvo em: Dados Gerados\META(1)_2023.xlsx
  Processando 'META(2024)'... 

...

(colunas: 298) ✅
    💾 Salvo em: Dados Gerados\META(2027).xlsx
  Processando 'iNEXECUTADO (pendências até 24)'... 

c:\Users\carlos.ramos\OneDrive - Presidência da República\Área de Trabalho\Planilhas\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


(colunas: 93) ✅
    💾 Salvo em: Dados Gerados\iNEXECUTADO_(pendências_até_24).xlsx
  Processando 'À EXECUTAR (2026 DIANTE)'... 

c:\Users\carlos.ramos\OneDrive - Presidência da República\Área de Trabalho\Planilhas\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


(colunas: 105) ✅
    💾 Salvo em: Dados Gerados\À_EXECUTAR_(2026_DIANTE).xlsx


🔗 Concatenando PLAN_EXEC + META(2023) + META(2024)...
✅ Concatenado e salvo em: Dados Gerados\PLAN_EXEC_ATÉ_ANO_ANTERIOR_(24).xlsx

📄 Gerando arquivos extras...
✅ 2025(2).xlsx gerado.
✅ PER(2).xlsx gerado.

🎯 Processamento concluído com sucesso!
Todos os arquivos estão em 'Dados Gerados/'.
